# Mini-Project 3: MNIST From Scratch to CNN

**Goal:** Build 3 progressively better models for MNIST digit classification and track accuracy at each step to see the improvement.

**Progression:**
1. **Logistic Regression** — a single linear layer (baseline)
2. **Multi-Layer Perceptron (MLP)** — add hidden layers
3. **Regularized MLP** — add Dropout and BatchNorm
4. **Convolutional Neural Network (CNN)** — exploit 2D image structure

**Estimated time:** 2-3 hours

---

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Device detection
device = torch.device('cuda' if torch.cuda.is_available() 
                      else 'mps' if torch.backends.mps.is_available() 
                      else 'cpu')
print(f"Using device: {device}")

## Load MNIST Dataset

In [ ]:
# Load MNIST dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean and std
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

print(f"Training samples: {len(train_dataset):,}")
print(f"Test samples:     {len(test_dataset):,}")
print(f"Image shape:      {train_dataset[0][0].shape}")
print(f"Classes:          {train_dataset.classes}")

# Show a sample grid of 25 images
fig, axes = plt.subplots(5, 5, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(f"Label: {label}", fontsize=10)
    ax.axis('off')
plt.suptitle("Sample MNIST Images", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Results Tracker

In [ ]:
results = {}

def log_model(name, train_acc, test_acc, n_params):
    results[name] = {'Train Acc': train_acc, 'Test Acc': test_acc, 'Parameters': n_params}
    print(f"{name}: Train={train_acc:.2%}, Test={test_acc:.2%}, Params={n_params:,}")

## Training Helper Functions

These are provided so you can focus on building the model architectures. Use `train_model()` to train any model, and `count_params()` to count parameters.

In [ ]:
def train_model(model, train_loader, test_loader, epochs=10, lr=0.001):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        if (epoch + 1) % 2 == 0 or epoch == 0:
            train_acc = evaluate(model, train_loader)
            test_acc = evaluate(model, test_loader)
            print(f"Epoch {epoch+1}/{epochs} — Loss: {total_loss/len(train_loader):.4f}, Train: {train_acc:.2%}, Test: {test_acc:.2%}")
    
    return evaluate(model, train_loader), evaluate(model, test_loader)

def evaluate(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            _, predicted = outputs.max(1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()
    return correct / total

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

---

## Part 1: Logistic Regression Baseline

A single linear layer — the simplest possible neural net. This maps each flattened 28x28=784 pixel image directly to 10 class scores.

In [ ]:
# TODO: Define a model with just nn.Linear(784, 10)
# TODO: Train for 5 epochs
# TODO: Evaluate and log results
# Hint: You'll need to flatten the 28x28 images to 784 vectors
#       Use nn.Flatten() as the first layer, or x.view(x.size(0), -1) in forward()

# Example structure:
# class LogisticRegression(nn.Module):
#     def __init__(self):
#         super().__init__()
#         ...
#     def forward(self, x):
#         ...

# model_lr = LogisticRegression()
# print(f"Logistic Regression params: {count_params(model_lr):,}")
# train_acc, test_acc = train_model(model_lr, train_loader, test_loader, epochs=5)
# log_model("Logistic Regression", train_acc, test_acc, count_params(model_lr))

---

## Part 2: Multi-Layer Perceptron

Add hidden layers and see the improvement. Non-linear activation functions (ReLU) allow the network to learn more complex decision boundaries.

In [ ]:
# TODO: Define MLP: 784 -> 256 -> ReLU -> 128 -> ReLU -> 10
# TODO: Train for 10 epochs
# TODO: Evaluate and log results

# model_mlp = MLP()
# print(f"MLP params: {count_params(model_mlp):,}")
# train_acc, test_acc = train_model(model_mlp, train_loader, test_loader, epochs=10)
# log_model("MLP", train_acc, test_acc, count_params(model_mlp))

---

## Part 3: MLP with Regularization

Add Dropout and BatchNorm. BatchNorm stabilizes training by normalizing layer inputs. Dropout prevents overfitting by randomly zeroing neurons during training.

In [ ]:
# TODO: Same MLP but add:
#   - nn.BatchNorm1d after each hidden layer (before ReLU)
#   - nn.Dropout(0.3) after each ReLU
# Architecture: 784 -> 256 -> BN -> ReLU -> Dropout -> 128 -> BN -> ReLU -> Dropout -> 10
# TODO: Train for 10 epochs
# TODO: Evaluate and log results

# model_reg = RegularizedMLP()
# print(f"Regularized MLP params: {count_params(model_reg):,}")
# train_acc, test_acc = train_model(model_reg, train_loader, test_loader, epochs=10)
# log_model("Regularized MLP", train_acc, test_acc, count_params(model_reg))

---

## Part 4: Simple CNN

Now exploit the 2D structure of images. Convolutional layers detect local patterns (edges, curves, shapes) regardless of their position in the image — this is the key insight that makes CNNs so powerful for vision tasks.

In [ ]:
# TODO: Define CNN:
#   Conv2d(1, 32, 3, padding=1) -> ReLU -> MaxPool2d(2)       # 28x28 -> 14x14
#   Conv2d(32, 64, 3, padding=1) -> ReLU -> MaxPool2d(2)      # 14x14 -> 7x7
#   Flatten -> Linear(64*7*7, 128) -> ReLU -> Dropout(0.5) -> Linear(128, 10)
# TODO: Train for 10 epochs
# TODO: Evaluate and log results
# Note: CNN takes the original 1x28x28 images — no flattening at the input!

# model_cnn = CNN()
# print(f"CNN params: {count_params(model_cnn):,}")
# train_acc, test_acc = train_model(model_cnn, train_loader, test_loader, epochs=10)
# log_model("CNN", train_acc, test_acc, count_params(model_cnn))

---

## Part 5: Compare All Models

Run this after completing all four models above to see the progression.

In [ ]:
results_df = pd.DataFrame(results).T
print(results_df.to_string())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
results_df[['Train Acc', 'Test Acc']].plot(kind='bar', ax=ax1, edgecolor='black')
ax1.set_title('Accuracy Comparison')
ax1.set_ylabel('Accuracy')
ax1.set_ylim(0.9, 1.0)
ax1.legend()

ax2.bar(results_df.index, results_df['Parameters'], edgecolor='black')
ax2.set_title('Parameter Count')
ax2.set_ylabel('# Parameters')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---

## Part 6: Error Analysis

Using your best model (CNN), find and visualize misclassified images. Look for patterns in the errors.

In [ ]:
# TODO: Using your best model (CNN), find misclassified images
# Show a grid of 20 misclassified examples with true vs predicted labels
# Are there patterns? (e.g., 4s confused with 9s?)

# Hints:
# 1. Run the model on the test set and collect (image, true_label, pred_label) for wrong predictions
# 2. Use a subplot grid to display them
# 3. Title each subplot with "True: X, Pred: Y"

---

## What I Learned

- **Logistic Regression:** ...
- **Adding hidden layers (MLP):** ...
- **Regularization (BatchNorm + Dropout):** ...
- **CNNs vs MLPs for images:** ...
- **The accuracy progression was:** ...
- **Surprising finding from error analysis:** ...
- **One thing I want to explore further:** ...